# JODI Dashboard

Three-dropdown interactive view of the consolidated JODI database:

1. **Product** — secondary or primary series (e.g. TOTPRODS, GASDIES).
2. **Country / region** — single country, a region (including consolidated **Asia Pacific**), or `Global Total`.
3. **Metric** — `Demand`, `Ending stocks`, or `Days of forward demand cover`.

> **Workflow**: Kernel restart → **Clear All Outputs** → run **Sections 1 → 2 → 3** once → run each UI section (**5–10**, **12**) once. Re-running UI cells duplicates widgets. **Section 11** is a static table (run after Section 1). **Section 12** compares demand YoY/MoM across products.

> **Regional drivers**: use `11_jodi_regional_drivers.ipynb` for demand/stocks decomposition (edit `GEOGRAPHY` in Setup).

> **One-time setup**: `pip install ipywidgets>=8.0` (in `requirements.txt`).

> **Scaling later**: logic lives in `analytics/jodi_dashboard.py`; a Streamlit/Dash port is a different UI shell over the same functions.


## 1. Setup & data load

In [1]:
"""Imports + parquet load. Same pattern as 04_jodi_explore.ipynb so the notebook
is self-contained and can be opened without depending on sibling notebooks."""

from pathlib import Path
import sys

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "update_jodi.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "scripts" / "update_jodi.py").exists():
            return candidate / "country_oil_scraper"
    raise RuntimeError(f"Could not locate project root from cwd: {here}")


PROJECT_ROOT = _resolve_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "jodi"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from analytics import seasonality_by_year_chart

df_sec = pd.read_parquet(PROCESSED_DIR / "jodi_secondary.parquet")
df_pri = pd.read_parquet(PROCESSED_DIR / "jodi_primary.parquet")

# Build an ISO-code -> country-name lookup once. country_name was joined onto each row
# by the processor, so any non-null pairing is canonical.
COUNTRY_NAMES: dict[str, str] = (
    pd.concat([df_sec[["ref_area", "country_name"]], df_pri[["ref_area", "country_name"]]])
      .dropna()
      .drop_duplicates("ref_area")
      .set_index("ref_area")["country_name"]
      .astype(str)
      .to_dict()
)
print(f"Secondary rows: {len(df_sec):,}")
print(f"Primary   rows: {len(df_pri):,}")
print(f"Country names known for {len(COUNTRY_NAMES)} ISO codes")

Secondary rows: 15,491,924
Primary   rows: 6,879,600
Country names known for 118 ISO codes


In [2]:
blanks = df_sec[(df_sec["unit_measure"] != "CONVBBL") & (~df_sec["obs_value"].isna())]

latest_per_country = (
    blanks.groupby("country_name", as_index=False)["date"]
          .max()
          .sort_values("date", ascending=False)
)
#latest_per_country.to_csv('latest_info.csv')



C:\Users\luiscarlos.gaitan\AppData\Local\Temp\ipykernel_57112\2906519767.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  blanks.groupby("country_name", as_index=False)["date"]


## 2. Dashboard core (import)

Logic lives in `analytics/jodi_dashboard.py`. Run **once** after Section 1.


In [3]:
"""Bind JODI dashboard core from analytics.jodi_dashboard."""

from analytics.jodi_dashboard import (
    configure,
    PRODUCTS_PRIMARY,
    PRODUCTS_SECONDARY,
    PRODUCT_TO_DATASET,
    REGION_MAP,
    CONSOLIDATED_ASIA_PACIFIC,
    REGION_ORDER,
    GLOBAL_KEY,
    REGION_PREFIX,
    METRIC_LABELS,
    ALL_DASHBOARD_PRODUCT_CODES,
    SEASONALITY_YEARS_BACK,
    SNAPSHOT_HISTORY_YEARS,
    DRIVER_LAG_MONTHS,
    SECONDARY_DEMAND_PRODUCT_CODES,
    resolve_geography,
    metric_spec,
    get_dashboard_series,
    assess_reporter_freshness,
    render_chart,
    get_all_products_series,
    build_product_snapshot_table,
    build_product_snapshot_tables,
    snapshot_month_options,
    style_product_snapshot_table,
    build_regional_driver_summary,
    build_country_contribution_table,
    build_common_panel,
    build_product_change_summary,
    plot_product_change_bars,
    style_regional_driver_summary,
    style_country_contribution_table,
    build_seasonal_frame,
    render_seasonal_chart,
    render_seasonality_by_year_all_products,
    product_label as _product_label,
)

configure(df_sec, df_pri, COUNTRY_NAMES)

print(
    f"Dashboard core configured "
    f"({len(SECONDARY_DEMAND_PRODUCT_CODES)} secondary products)"
)


Dashboard core configured (9 secondary products)


## 3. Widget helpers

Dropdown lists, debounced callbacks, and lag controls. **Required** before Sections 5–10.


In [4]:
"""Widget helpers for Sections 5–10. Run once after Section 2 (kernel restart → 1 → 2 → 3)."""

import threading


def _bind_once(widget, handler, *, names: str = "value", tag: str = "default") -> None:
    """Register a widget callback without stacking duplicates on cell re-run."""
    registry = getattr(widget, "_dashboard_handlers", None)
    if registry is None:
        registry = {}
        widget._dashboard_handlers = registry
    old = registry.get(tag)
    if old is not None:
        try:
            widget.unobserve(old, names=names)
        except (ValueError, KeyError):
            pass
    widget.observe(handler, names=names)
    registry[tag] = handler


def _debounce(delay_ms: int = 250):
    """Coalesce rapid widget callbacks into a single render (avoids stacked Output)."""

    def decorator(fn):
        state = {"timer": None, "generation": 0}

        def wrapped(*args, **kwargs):
            state["generation"] += 1
            generation = state["generation"]
            if state["timer"] is not None:
                state["timer"].cancel()

            def fire() -> None:
                if generation == state["generation"]:
                    fn(*args, **kwargs)

            state["timer"] = threading.Timer(delay_ms / 1000.0, fire)
            state["timer"].start()

        wrapped.__name__ = fn.__name__
        wrapped.__doc__ = fn.__doc__
        return wrapped

    return decorator


def _product_dropdown_options() -> list[tuple[str, str]]:
    headline = [
        ("Total oil products (TOTPRODS)", "TOTPRODS"),
        ("Total crude (TOTCRUDE)", "TOTCRUDE"),
    ]
    rest_secondary = sorted(
        [(f"{label} ({code})", code) for code, label in PRODUCTS_SECONDARY.items()
         if code != "TOTPRODS"],
        key=lambda x: x[0].lower(),
    )
    rest_primary = sorted(
        [(f"{label} ({code})", code) for code, label in PRODUCTS_PRIMARY.items()
         if code != "TOTCRUDE"],
        key=lambda x: x[0].lower(),
    )
    return headline + rest_secondary + rest_primary


def _country_dropdown_options() -> list[tuple[str, str]]:
    opts: list[tuple[str, str]] = [("Global Total", GLOBAL_KEY)]
    opts += [(f"{r} (region)", REGION_PREFIX + r) for r in REGION_ORDER]
    present_codes = (
        set(df_sec["ref_area"].astype(str).unique())
        | set(df_pri["ref_area"].astype(str).unique())
    )
    countries = sorted(
        [(COUNTRY_NAMES.get(code, code), code) for code in present_codes if code],
        key=lambda x: x[0].lower(),
    )
    return opts + countries


def _metric_dropdown_options() -> list[tuple[str, str]]:
    return [(label, code) for code, label in METRIC_LABELS.items()]


def _secondary_product_dropdown_options() -> list[tuple[str, str]]:
    headline = [("Total oil products (TOTPRODS)", "TOTPRODS")]
    rest = sorted(
        [(f"{label} ({code})", code) for code, label in PRODUCTS_SECONDARY.items()
         if code != "TOTPRODS"],
        key=lambda x: x[0].lower(),
    )
    return headline + rest


exclude_lagging_cb = widgets.Checkbox(
    value=False,
    description="Exclude lagging reporters from group totals",
    indent=False,
)
lag_months_slider = widgets.IntSlider(
    value=2,
    min=0,
    max=6,
    step=1,
    description="Lag ≥ months:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)


def _exclusion_kwargs() -> dict:
    return {
        "exclude_lagging": exclude_lagging_cb.value,
        "lag_months": int(lag_months_slider.value),
    }


## 5. Dashboard (time series)

Three dropdowns drive a single live-updating time-series chart. Change any selection — no need to re-run the cell.

## 6. Seasonality by year (all products)

Standalone panel with its own **Country** and **Metric** dropdowns (independent of Section 5). One small-multiple per product via `analytics.seasonality_by_year_chart` — last six calendar years, latest year in red.

In [5]:
"""Section 5 — time series chart."""

product_dd = widgets.Dropdown(
    options=_product_dropdown_options(),
    value="TOTPRODS",
    description="Product:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
country_dd = widgets.Dropdown(
    options=_country_dropdown_options(),
    value=GLOBAL_KEY,
    description="Country:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
metric_dd = widgets.Dropdown(
    options=_metric_dropdown_options(),
    value="demand",
    description="Metric:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)

ts_out = widgets.Output()


@_debounce(200)
def _render_timeseries(*_change) -> None:
    with ts_out:
        ts_out.clear_output(wait=True)
        render_chart(
            product_dd.value, country_dd.value, metric_dd.value,
            **_exclusion_kwargs(),
        ).show()


ui = widgets.VBox([
    product_dd, country_dd, metric_dd,
    exclude_lagging_cb, lag_months_slider,
])

for _w in (product_dd, country_dd, metric_dd, exclude_lagging_cb, lag_months_slider):
    _bind_once(_w, _render_timeseries, tag="sec5_timeseries")

display(ui, ts_out)
_render_timeseries()


Output()

## 8. Seasonal pattern (envelope)

A second view of the same data, organised by **calendar month** instead of by full
date. Gray bands show the previous five calendar years; coloured lines overlay years you toggle.

Tracks the **Product / Country / Metric** dropdowns from **Section 5** (run that cell first) — change a Section 5 dropdown and this chart updates.

What you'll see:

- Light-gray **min-max envelope** across the previous 5 calendar years
- Darker gray **p25-p75 band** and dashed **median**
- One **bold coloured line per year** you tick (current = green, previous = red)
- Partial current-year months appear as gaps, not interpolated segments

In [6]:
"""Section 6 — seasonality by year (all products)."""

seas_country_dd = widgets.Dropdown(
    options=_country_dropdown_options(),
    value=GLOBAL_KEY,
    description="Country:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
seas_metric_dd = widgets.Dropdown(
    options=_metric_dropdown_options(),
    value="demand",
    description="Metric:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)

seas_out = widgets.Output()


@_debounce(300)
def _render_seasonality_panel(*_change) -> None:
    with seas_out:
        seas_out.clear_output(wait=True)
        render_seasonality_by_year_all_products(
            seas_country_dd.value, seas_metric_dd.value,
            **_exclusion_kwargs(),
        ).show()


seas_ui = widgets.VBox([seas_country_dd, seas_metric_dd])

for _w in (seas_country_dd, seas_metric_dd):
    _bind_once(_w, _render_seasonality_panel, tag="sec6_seasonality")

display(seas_ui, seas_out)
_render_seasonality_panel()


## 7. Product snapshot — YoY vs 5-year range

For the selected **country/region** and **reference month**, one table per metric:

- **YoY** — level vs the same calendar month last year (`yoy_change`, `yoy_pct`)
- **5y band** — min / max / median for that month over the prior five years (excludes the reference year; same window as the seasonal envelope)
- **vs 5y range** — whether the current month is below the 5y low, above the 5y high, or where it sits on the min–max scale (`pct_in_5y_range`)

Both **Demand** and **Ending stocks** tables update together when you change country or month.

In [7]:
"""Section 7 — product snapshot (YoY vs 5y band)."""

from IPython.display import HTML

snap_country_dd = widgets.Dropdown(
    options=_country_dropdown_options(),
    value=GLOBAL_KEY,
    description="Country:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
snap_month_dd = widgets.Dropdown(
    options=snapshot_month_options(GLOBAL_KEY),
    description="Month:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
snap_refresh_btn = widgets.Button(description="Refresh", button_style="primary")
snap_status_lbl = widgets.Label(value="")

snap_out = widgets.Output()


def _refresh_snapshot_month_options() -> None:
    opts = snapshot_month_options(snap_country_dd.value)
    with snap_month_dd.hold_trait_notifications():
        snap_month_dd.options = opts
        if opts:
            snap_month_dd.value = opts[0][1]


def _render_snapshot_tables(_=None) -> None:
    geography = snap_country_dd.value
    ref_year, ref_month = snap_month_dd.value
    _, geo_label = resolve_geography(geography)
    month_lbl = f"{ref_year}-{ref_month:02d}"

    with snap_out:
        snap_out.clear_output(wait=True)
        snap_status_lbl.value = "Loading…"
        display(HTML(f"<h4>{geo_label} — {month_lbl}</h4>"))

        tables = build_product_snapshot_tables(
            geography,
            reference_year=ref_year,
            reference_month=ref_month,
            **_exclusion_kwargs(),
        )
        for metric, title in (
            ("demand", "Demand (kb/d)"),
            ("stocks", "Ending stocks (mbbl)"),
        ):
            tbl = tables[metric]
            display(HTML(f"<b>{title}</b>"))
            if tbl.empty or tbl["current"].isna().all():
                display(HTML("<p><i>No data for this selection</i></p>"))
            else:
                display(style_product_snapshot_table(tbl))
        snap_status_lbl.value = "Done."


def _on_snap_country_change(_change) -> None:
    _refresh_snapshot_month_options()


_bind_once(snap_country_dd, _on_snap_country_change, tag="sec7_snapshot_country")
snap_refresh_btn.on_click(_render_snapshot_tables)

snap_ui = widgets.VBox([
    snap_country_dd,
    snap_month_dd,
    widgets.HBox([snap_refresh_btn, snap_status_lbl]),
])

display(snap_ui, snap_out)
_refresh_snapshot_month_options()
_render_snapshot_tables()


In [8]:
"""Year-toggle widgets + reactive wiring for the seasonal chart.

We deliberately REUSE the existing ``product_dd`` / ``country_dd`` / ``metric_dd``
dropdowns from the cell at the top of section 5, so the time-series chart and
the seasonal chart always show the same selection. The user only has to learn
the dropdowns once.

The extra control here is a row of checkboxes, one per calendar year present in
the data. Ticking a box adds that year as an overlay line on the seasonal chart.
"""

# Build the master list of years present anywhere in either parquet. We use
# *all* years so the checkbox row is stable across selections - toggling a year
# that happens to have no data for the current product/country is a no-op (the
# renderer skips it), which is friendlier than checkboxes appearing / disappearing.
_all_years_seasonal: list[int] = sorted(
    set(pd.to_datetime(df_sec["date"]).dt.year.unique())
    | set(pd.to_datetime(df_pri["date"]).dt.year.unique())
)
_latest_year_seasonal = int(max(_all_years_seasonal))

# One ipywidgets.Checkbox per calendar year. Defaults: ON for the current and
# immediate-previous year (matches the 2025/2026 highlighting in your reference
# image); OFF for everything else so the chart isn't crowded by default.
year_checkboxes: dict[int, widgets.Checkbox] = {
    year: widgets.Checkbox(
        value=(year in (_latest_year_seasonal, _latest_year_seasonal - 1)),
        description=str(year),
        # indent=False removes the indentation ipywidgets adds by default for
        # the (now hidden) description label - we want compact checkboxes.
        indent=False,
        layout=widgets.Layout(width="80px"),
    )
    for year in _all_years_seasonal
}

# Compose the picker: a small caption above a horizontal row of checkboxes.
# ``flex_flow="row wrap"`` lets the row break onto a second line on narrower
# screens instead of overflowing.
year_picker = widgets.VBox([
    widgets.Label("Show years (toggle to overlay on the seasonal chart):"),
    widgets.HBox(
        list(year_checkboxes.values()),
        layout=widgets.Layout(flex_flow="row wrap"),
    ),
])

# A dedicated Output widget that the callback writes the figure into. Using an
# Output widget (rather than calling fig.show() directly) is what lets us
# ``clear_output(wait=True)`` so the chart doesn't visually "blink" between renders.
seasonal_out = widgets.Output()


@_debounce(200)
def _render_seasonal_now(*_change) -> None:
    """Observer that re-renders the seasonal chart from current widget state.

    Fires on any product / geography / metric / checkbox change. We ignore the
    change-event payload (``*_change``) and just *read* the latest value from
    every widget directly - simpler and easier to reason about than trying to
    dispatch on event type.
    """
    # Collect ticked years in calendar order. The renderer also sorts internally
    # for stable legend order, but doing it here costs nothing and keeps the
    # passed argument predictable when debugging.
    selected = [year for year, cb in year_checkboxes.items() if cb.value]

    # ``with seasonal_out:`` redirects everything displayed inside the block into
    # the Output widget instead of the notebook's main output area.
    with seasonal_out:
        seasonal_out.clear_output(wait=True)
        render_seasonal_chart(
            product=product_dd.value,
            geography=country_dd.value,
            metric=metric_dd.value,
            selected_years=selected,
            **_exclusion_kwargs(),
        ).show()


for _w in (product_dd, country_dd, metric_dd):
    _bind_once(_w, _render_seasonal_now, tag="sec8_seasonal_envelope")
for _year, _cb in year_checkboxes.items():
    _bind_once(_cb, _render_seasonal_now, tag=f"sec8_year_{_year}")

display(year_picker, seasonal_out)
_render_seasonal_now()


Output()

## 9. Reporter freshness (data lag)

For **multi-country** selections, compares each country's latest reporting month for the chosen **product + metric** against the group's most recent peer.

- **flag_lagging** — no data, or more than *N* months behind the group's latest print (slider in Section 5).
- Enable **Exclude lagging reporters from group totals** (Section 5) to remove them from regional/global sums in Sections 5–8 so a demand dip is not confused with missing data.

Single-country picks show an empty table (no peers to compare).

In [9]:
"""Section 9 — reporter freshness."""

from IPython.display import HTML

fresh_country_dd = widgets.Dropdown(
    options=_country_dropdown_options(),
    value=GLOBAL_KEY,
    description="Country:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
fresh_product_dd = widgets.Dropdown(
    options=_product_dropdown_options(),
    value="TOTPRODS",
    description="Product:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
fresh_metric_dd = widgets.Dropdown(
    options=[("Demand", "demand"), ("Ending stocks", "stocks")],
    value="demand",
    description="Metric:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)

fresh_out = widgets.Output()


@_debounce(200)
def _render_freshness_table(*_change) -> None:
    geography = fresh_country_dd.value
    product = fresh_product_dd.value
    metric = fresh_metric_dd.value
    lag = int(lag_months_slider.value)
    _, geo_label = resolve_geography(geography)

    with fresh_out:
        fresh_out.clear_output(wait=True)
        codes, _ = resolve_geography(geography)
        if len(codes) <= 1:
            display(HTML(
                f"<p><i>{geo_label} is a single country — no peer lag comparison. "
                "Pick a region or Global Total.</i></p>"
            ))
            return

        tbl = assess_reporter_freshness(
            geography, product, metric, lag_months=lag,
        )
        n_lag = int(tbl["flag_lagging"].sum())
        peer = tbl["peer_latest"].iloc[0]
        peer_str = peer.strftime("%Y-%m") if pd.notna(peer) else "—"
        display(HTML(
            f"<h4>{geo_label} — {_product_label(product)} — {METRIC_LABELS[metric]}</h4>"
            f"<p>Group latest month: <b>{peer_str}</b> · "
            f"Lagging (≥{lag} mo behind): <b>{n_lag}</b> / {len(tbl)} countries</p>"
        ))
        show_cols = [
            "country", "ref_area", "latest_date", "peer_latest",
            "months_behind_peer", "flag_lagging",
        ]
        view = tbl[show_cols].copy()
        view["latest_date"] = view["latest_date"].dt.strftime("%Y-%m")
        view["peer_latest"] = view["peer_latest"].dt.strftime("%Y-%m")
        display(
            view.style.format({"months_behind_peer": "{:.0f}"})
            .background_gradient(subset=["months_behind_peer"], cmap="YlOrRd")
            .set_properties(**{"text-align": "left"})
        )
        if exclude_lagging_cb.value and n_lag:
            excluded = ", ".join(
                tbl.loc[tbl["flag_lagging"], "country"].head(8).astype(str)
            )
            display(HTML(
                f"<p><i>Excluded from group totals: {excluded}"
                f"{'…' if n_lag > 8 else ''}</i></p>"
            ))


fresh_ui = widgets.VBox([fresh_country_dd, fresh_product_dd, fresh_metric_dd])

for _w in (fresh_country_dd, fresh_product_dd, fresh_metric_dd):
    _bind_once(_w, _render_freshness_table, tag="sec9_freshness")

display(fresh_ui, fresh_out)
_render_freshness_table()


## 10. Regional demand drivers (secondary)

Decomposes **secondary demand** changes into seasonal (**YoY**, same calendar month) and
sequential (**MoM**, prior month) drivers using a **balanced panel** only:

- Panel members must have valid demand in the reference month, the prior-year same month,
  and the prior month.
- **Lag = 0** — only countries at the group's latest print are eligible (stricter than
  Section 5's slider; fixed here so panel totals are apples-to-apples).
- Country table shows **panel members only** (no greyed-out excluded rows).

Layer 1: all secondary products for the selected region. Layer 2: country contributions
for the selected product.

In [10]:
"""Section 10 — regional demand drivers (button refresh)."""

from IPython.display import HTML

_driver_summary_cache: dict = {"geography": None, "table": None}

driver_country_dd = widgets.Dropdown(
    options=_country_dropdown_options(),
    value=REGION_PREFIX + CONSOLIDATED_ASIA_PACIFIC,
    description="Region:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
driver_product_dd = widgets.Dropdown(
    options=_secondary_product_dropdown_options(),
    value="TOTPRODS",
    description="Product:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
driver_refresh_btn = widgets.Button(description="Refresh", button_style="primary")
driver_status_lbl = widgets.Label(value="")

driver_out = widgets.Output()


def _load_driver_summary(geography: str):
    if _driver_summary_cache["geography"] != geography:
        _driver_summary_cache["geography"] = geography
        _driver_summary_cache["table"] = build_regional_driver_summary(
            geography, lag_months=DRIVER_LAG_MONTHS,
        )
    return _driver_summary_cache["table"]


def _render_driver_panel(_=None) -> None:
    geography = driver_country_dd.value
    product = driver_product_dd.value
    _, geo_label = resolve_geography(geography)

    with driver_out:
        driver_out.clear_output(wait=True)
        driver_status_lbl.value = "Loading…"
        codes, _ = resolve_geography(geography)
        if len(codes) <= 1:
            display(HTML(
                f"<p><i>{geo_label} is a single country — pick a region or Global Total "
                "for a multi-country driver decomposition.</i></p>"
            ))
            driver_status_lbl.value = ""
            return

        summary = _load_driver_summary(geography)
        contrib = build_country_contribution_table(
            geography, product, lag_months=DRIVER_LAG_MONTHS,
        )

        ref_row = summary.loc[summary["product_code"] == product]
        ref_month = ref_row["month"].iloc[0] if not ref_row.empty else "—"
        panel_n = int(ref_row["panel_n"].iloc[0]) if not ref_row.empty else 0

        display(HTML(
            f"<h4>{geo_label} — secondary demand drivers</h4>"
            f"<p>Balanced panel at lag = {DRIVER_LAG_MONTHS} mo: "
            f"same countries in reference, YoY, and MoM months, all at the group "
            f"latest print. Reference month varies by product. "
            f"For a focused APAC workflow see "
            f"<code>11_jodi_regional_drivers.ipynb</code>.</p>"
        ))

        show_summary = summary[[
            "product", "month", "level_kb_d", "yoy_change", "yoy_pct",
            "mom_change", "mom_pct", "vs_5y_range", "panel_n", "excluded_n",
        ]]
        display(HTML("<h5>All products — regional change (panel totals)</h5>"))
        display(style_regional_driver_summary(show_summary))

        display(HTML(
            f"<h5>{_product_label(product)} — country contributions "
            f"({panel_n} panel members, ref {ref_month})</h5>"
        ))
        if contrib.empty:
            display(HTML("<p><i>No balanced panel for this product.</i></p>"))
        else:
            display(style_country_contribution_table(contrib))
        driver_status_lbl.value = "Done."


driver_refresh_btn.on_click(_render_driver_panel)

driver_ui = widgets.VBox([
    driver_country_dd,
    driver_product_dd,
    widgets.HBox([driver_refresh_btn, driver_status_lbl]),
])

display(driver_ui, driver_out)
_render_driver_panel()


Output()

## 11. Stock change since Feb 2026 (all countries)

Cross-country table of **JODI closing stocks** (`CLOSTLV`, KBBL) change from **Feb 2026** to the **latest global reporting month** after that baseline.

Products use the non-overlapping kerosene split: `X_OTHKERO` (other kerosene) and `JETKERO` (jet). Values are **mbbl** (change in million barrels). Rows with no product data are dropped; countries with at least one non-null product change are kept. A **Total (all countries)** row at the bottom sums each product column across countries.

In [11]:
"""Section 11 — stock change since Feb 2026 (countries × products)."""

STOCK_WAR_PRODUCTS: list[tuple[str, str]] = [
    ("LPG", "LPG"),
    ("NAPHTHA", "Naphtha"),
    ("GASOLINE", "Gasoline"),
    ("X_OTHKERO", "Kerosene"),
    ("JETKERO", "Jet fuel"),
    ("GASDIES", "Diesel"),
    ("RESFUEL", "Residual fuel"),
    ("ONONSPEC", "Other"),
]
PRODUCT_CODES = [code for code, _ in STOCK_WAR_PRODUCTS]
COL_LABELS = [label for _, label in STOCK_WAR_PRODUCTS]

BASELINE = pd.Timestamp("2026-02-01")

stocks_war = df_sec[
    (df_sec["flow_breakdown"] == "CLOSTLV")
    & (df_sec["unit_measure"] == "KBBL")
    & (df_sec["value_status"] == "valid")
    & (df_sec["energy_product"].isin(PRODUCT_CODES))
].copy()

stocks_war["date"] = pd.to_datetime(stocks_war["date"])
stocks_war["ref_area"] = stocks_war["ref_area"].astype(str)

post_baseline = stocks_war[stocks_war["date"] > BASELINE]
if post_baseline.empty:
    print("[skip] No CLOSTLV rows after Feb 2026 in jodi_secondary.parquet")
else:
    global_latest = post_baseline["date"].max()

    base = stocks_war[stocks_war["date"] == BASELINE][
        ["ref_area", "energy_product", "obs_value"]
    ].rename(columns={"obs_value": "base_kb"})

    latest = stocks_war[stocks_war["date"] == global_latest][
        ["ref_area", "energy_product", "obs_value"]
    ].rename(columns={"obs_value": "latest_kb"})

    chg = base.merge(latest, on=["ref_area", "energy_product"], how="inner")
    chg["change_mbbl"] = (chg["latest_kb"] - chg["base_kb"]) / 1000.0

    wide = (
        chg.pivot(index="ref_area", columns="energy_product", values="change_mbbl")
        .reindex(columns=PRODUCT_CODES)
    )
    wide.columns = COL_LABELS
    wide = wide.dropna(how="all")
    wide["Total"] = wide.sum(axis=1, min_count=1)
    wide.index = wide.index.map(lambda c: COUNTRY_NAMES.get(c, c))
    wide.index.name = "Country"

    out = wide.sort_values("Total").round(2)
    total_row = out.sum(min_count=1)
    total_row.name = "Total (all countries)"
    out = pd.concat([out, total_row.to_frame().T])

    print(
        f"Baseline: {BASELINE.strftime('%Y-%m')}  |  "
        f"Latest (global): {global_latest.strftime('%Y-%m')}  |  "
        f"Countries: {len(wide):,}"
    )
    display(out)

Baseline: 2026-02  |  Latest (global): 2026-05  |  Countries: 44


,LPG,Naphtha,Gasoline,Kerosene,Jet fuel,Diesel,Residual fuel,Other,Total
Saudi Arabia,-0.14,-1.05,-10.22,<NA>,0.0,-9.53,-1.41,0.45,-21.88
Netherlands (Kingdom of the),0.38,-1.58,-3.36,<NA>,1.73,-4.51,-4.75,-0.88,-12.97
Italy,-0.46,-0.11,-0.52,<NA>,-0.76,-0.38,-1.46,-1.47,-5.16
Germany,0.12,0.38,0.35,<NA>,-0.7,-3.19,-0.88,0.17,-3.76
Sweden,0.27,-0.03,-0.59,<NA>,-0.35,-1.31,-1.13,0.25,-2.89
Denmark,0.02,0.0,-0.1,<NA>,0.16,-1.1,-2.06,0.26,-2.81
Canada,-0.06,-0.29,-1.08,<NA>,0.65,-1.53,0.28,-0.71,-2.74
United Kingdom of Great Britain and Northern Ireland (the),-0.12,-0.02,-1.5,<NA>,-0.24,0.56,0.41,-0.44,-1.35
Estonia,-0.03,0.0,-0.63,<NA>,-0.24,-0.28,-0.16,0.0,-1.33
Hungary,0.06,0.27,-0.73,<NA>,0.05,-1.06,0.16,0.07,-1.18


In [12]:
# Re-run Section 11 first. Includes the Total (all countries) footer row.
_out = wide.sort_values("Total").round(2)
_footer = _out.sum(min_count=1)
_footer.name = "Total (all countries)"
pd.concat([_out, _footer.to_frame().T]).to_clipboard()

## 12. Demand by product — YoY & MoM

All **secondary demand** products in one view: current level vs **last year** (same calendar month) and **last month** (sequential), using one **shared balanced country panel** (TOTPRODS-anchored, `lag = 0`).

- **Table** — level, YoY/MoM change (kb/d and %), vs 5y range, panel size
- **Chart** — side-by-side horizontal bars: YoY change | MoM change

For per-country decomposition of one product, use **Section 10** (regional drivers).

In [13]:
"""Section 12 — demand by product (YoY & MoM)."""

import matplotlib.pyplot as plt
from IPython.display import HTML

_DEMAND_UNIT = "kb/d"
_CHANGE_POS = "#2ca02c"
_CHANGE_NEG = "#d62728"

demand_change_country_dd = widgets.Dropdown(
    options=_country_dropdown_options(),
    value=GLOBAL_KEY,
    description="Country:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
demand_change_refresh_btn = widgets.Button(description="Refresh", button_style="primary")
demand_change_status_lbl = widgets.Label(value="")

demand_change_out = widgets.Output()


def _render_demand_change_panel(_=None) -> None:
    geography = demand_change_country_dd.value
    _, geo_label = resolve_geography(geography)

    panel, _ry, _rm, month_label, excluded_n = build_common_panel(
        geography,
        lag_months=DRIVER_LAG_MONTHS,
        metric="demand",
        panel_mode="totprods_anchor",
    )

    with demand_change_out:
        demand_change_out.clear_output(wait=True)
        demand_change_status_lbl.value = "Loading…"

        summary = build_product_change_summary(
            geography,
            lag_months=DRIVER_LAG_MONTHS,
            metric="demand",
            panel_mode="totprods_anchor",
        )

        subtitle = (
            f"ref {month_label} · balanced panel n={len(panel)} "
            f"(TOTPRODS-anchored) · lag={DRIVER_LAG_MONTHS} mo"
        )
        if excluded_n:
            subtitle += f" · {excluded_n} excluded"

        display(HTML(
            f"<h4>{geo_label} — demand by product</h4>"
            f"<p>{subtitle}. YoY = same calendar month last year; "
            f"MoM = prior month.</p>"
        ))

        cols = [
            "product", "month", "level_kb_d", "yoy_change", "yoy_pct",
            "mom_change", "mom_pct", "vs_5y_range", "panel_n", "excluded_n",
        ]
        tbl = summary[cols].copy()

        if tbl.empty or tbl["level_kb_d"].isna().all():
            display(HTML("<p><i>No data for this selection</i></p>"))
        else:
            display(style_regional_driver_summary(tbl))
            fig = plot_product_change_bars(
                summary,
                title=f"{geo_label} — {METRIC_LABELS['demand']} by product",
                unit=_DEMAND_UNIT,
                pos_color=_CHANGE_POS,
                neg_color=_CHANGE_NEG,
            )
            plt.show()

        demand_change_status_lbl.value = "Done."


_bind_once(
    demand_change_country_dd,
    _render_demand_change_panel,
    tag="sec12_demand_change_country",
)
demand_change_refresh_btn.on_click(_render_demand_change_panel)

demand_change_ui = widgets.VBox([
    demand_change_country_dd,
    widgets.HBox([demand_change_refresh_btn, demand_change_status_lbl]),
])

display(demand_change_ui, demand_change_out)
_render_demand_change_panel()

Output()

## 13. Porting to Streamlit later

The data layer (`get_dashboard_series` + `render_chart`) is intentionally pure - no
widget imports, no notebook globals - so a Streamlit app is essentially:

```python
# app.py
import streamlit as st
from dashboard_core import (   # extract sections 2-4 of this notebook to a .py module
    PRODUCTS_PRIMARY, PRODUCTS_SECONDARY, REGION_MAP, REGION_ORDER,
    GLOBAL_KEY, REGION_PREFIX, METRIC_LABELS,
    get_dashboard_series, render_chart,
)

product   = st.selectbox("Product",  options=[...])  # same option lists as the notebook
geography = st.selectbox("Country",  options=[...])
metric    = st.selectbox("Metric",   options=list(METRIC_LABELS))

st.plotly_chart(render_chart(product, geography, metric), use_container_width=True)
```

Run with `streamlit run app.py`. The only "porting" work is moving sections 2-4 of this
notebook into a small `dashboard_core.py` module (so both notebook and Streamlit app
import it), and rebuilding the option lists with `st.selectbox` instead of
`ipywidgets.Dropdown`. No business logic changes.